In [ ]:
from trouver.helper.html import (
    add_HTML_tag_data_to_raw_text, add_space_to_lt_symbols_without_space, remove_html_tags_in_text, StrAndHTMLTagsWithIndices,
    HTMLTagWithIndices)
from trouver.machine_learning.tokenize.def_and_notat_token_classification import predict_and_mark_def_and_notats

In [ ]:
from fastcore.test import *

from transformers import pipeline, AutoModelForTokenClassification, AutoTokenizer
from bs4 import BeautifulSoup


### Miscellaneous marking formatting

The following is helper code for making predictions on the author's writing

In [ ]:
#| export machine_learning.tokenize.def_and_notat_token_classification

def latex_highlight_formatter(text: str, pred: HTMLTagWithIndices) -> str:
    r"""
    Formats definitions and notations with LaTeX highlighting commands.
    
    Rules:
    - Definitions: \hldef{text}
    - Notations:
        - $...$ -> \hl{$...$}
        - $$...$$ -> $$\hlin{...}$$
        - \begin{align*}...\end{align*} -> \hlalign{\begin{align*}...\end{align*}}
    """
    is_definition = 'definition' in pred.tag.attrs
    
    # --- Case 1: Definition ---
    if is_definition:
        return f"\\hldef{{{text}}}"
    
    # --- Case 2: Notation ---
    # We need to detect the type of math environment in `text`
    
    stripped_text = text.strip()
    
    # Subcase 2a: Display Math with $$...$$
    if stripped_text.startswith('$$') and stripped_text.endswith('$$'):
        # Extract content inside $$...$$
        # Content length is len(text) - 4 (for the two $$ pairs)
        # We need to handle potential whitespace, so we use indices based on the strip
        inner_content = stripped_text[2:-2] 
        return f"$$\\hlin{{{inner_content}}}$$"
    
    # Subcase 2b: Align Environment
    # Checks for \begin{align*} or \begin{align}
    if r'\begin{align' in text:
        return f"\\hlalign{{{text}}}"
    
    # Subcase 2c: Inline Math $...$ (Default for notations)
    # Note: We wrap the *entire* text (including $) in \hl{...}
    # User requirement: "if $blah$, then \hl{$blah$}"
    return f"\\hl{{{text}}}"

In [ ]:
#| notest

# 1. Load the Model
# Replace 'hyunjongkim/math-def-and-notat-token-classification' with your actual model path/ID
# We use 'token-classification' task. 'aggregation_strategy="simple"' helps merge B- and I- tags automatically.
model = AutoModelForTokenClassification.from_pretrained('hyunjongkimmath/def_and_notat_token_classification_model_modernbert_base')
tokenizer = AutoTokenizer.from_pretrained('hyunjongkimmath/def_and_notat_token_classification_model_modernbert_base')
def_notat_classifier = pipeline('ner', model=model, tokenizer=tokenizer)

# 2. Setup the Inputs
# Text with definition ("Galois group"), inline notation ($...$), and display notation ($$...$$)
text = (
    r"Let $L/K$ be a separable, normal algebraic field extension. The Galois group $\operatorname{Gal}(L/K)$ is defined as the automorphism group."
    "\n\n"
    r"Consider the sequence: $$ 0 \to A \to B \to C \to 0 $$"
)

# A dummy note object (required by the function for logging)
# note = SimpleNamespace(name="Example Note", path="Example.md")

# 3. Run Prediction and Formatting
formatted_text = predict_and_mark_def_and_notats(
    main_text=text,
    pipeline=def_notat_classifier,
    # note=note,
    formatter=latex_highlight_formatter,
    excessive_space_threshold=2
)

# 4. View Result
print(formatted_text)

# Expected Output (conceptual):
# The \hldef{Galois group} \hl{$\operatorname{Gal}(L/K)$} is defined by the kernel. 
# Consider the sequence: $$\hlin{ 0 \to A \to B \to C \to 0 }$$

Device set to use cpu
Compiling the model with `torch.compile` and using a `torch.cpu` device is not supported. Falling back to non-compiled mode.


Let $L/K$ be a separable, normal algebraic field extension. The \hldef{Galois group} \hl{$\operatorname{Gal}(L/K)$} is defined as the automorphism group.

Consider the sequence: $$ 0 \to A \to B \to C \to 0 $$


In [ ]:
#| hide
# --- Setup ---
soup = BeautifulSoup("", 'html.parser')

def make_pred(tag_type, text):
    # Helper to create a dummy prediction
    attrs = {tag_type: ""}
    tag = soup.new_tag("span", **attrs)
    # Indices don't matter for the formatter, only the text passed to it
    return HTMLTagWithIndices(tag, 0, 0)

# --- Test 1: Definition ---
text_def = "Galois group"
pred_def = make_pred("definition", text_def)
assert latex_highlight_formatter(text_def, pred_def) == r"\hldef{Galois group}"

# --- Test 2: Inline Notation ---
text_inline = "$x$"
pred_not = make_pred("notation", text_inline)
assert latex_highlight_formatter(text_inline, pred_not) == r"\hl{$x$}"

# --- Test 3: Display Notation ($$) ---
text_display = "$$x^2$$"
pred_not = make_pred("notation", text_display)
# Expect: $$\hlin{x^2}$$
assert latex_highlight_formatter(text_display, pred_not) == r"$$\hlin{x^2}$$"

# Test 3b: Display with whitespace
text_display_ws = "$$  y^2  $$"
# Expect: $$\hlin{  y^2  }$$  (Inner content preserved)
assert latex_highlight_formatter(text_display_ws, pred_not) == r"$$\hlin{  y^2  }$$"

# --- Test 4: Align Notation ---
text_align = r"""\begin{align*}
x &= y \\
y &= z
\end{align*}"""
pred_not = make_pred("notation", text_align)

expected_align = r"\hlalign{" + text_align + "}"
assert latex_highlight_formatter(text_align, pred_not) == expected_align

print("All latex_highlight_formatter tests passed!")

All latex_highlight_formatter tests passed!
